# M54 controlled Vehicle/Pedestrian MonoDGP adaptation

This is an accuracy-first, ground-truth-only transfer-learning experiment from the verified M53 checkpoint. The MonoDGP ResNet50 and transformer graph stays unchanged. Run setup through the CUDA training smoke, then stop and return m54_training_smoke.json before starting the 100-epoch training cell. Distillation, temperature tuning, compression, and iPhone qualification are not part of M54.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, re, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M54')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m54')
M53_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodgp_m53')
M53_GATE=M53_ROOT/'m53_monodgp_reference_gate.json'
M53_MANIFEST=M53_ROOT/'m53_monodgp_reference_manifest.json'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/challengers/monodgp_m54')
RUN_NAME='monodgp_m54_vehicle_pedestrian_gt'
RUN_DIR=OUTPUT_ROOT/RUN_NAME
MAX_EPOCHS=100
SWEEP_EPOCHS=[5,10,15,20,30,40,50,60,70,80,90,100]
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]
    print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])


In [ ]:
# Fetch exact sources, apply audited compatibility/training patches, and build the CUDA extension.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','gdown','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','thop','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py',
    'lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py',
    'lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py',
    'lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)


In [ ]:
# Create a canonical Chen-split KITTI view without copying the image tree.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for required in (M53_GATE,M53_MANIFEST):
    if not required.is_file():
        raise FileNotFoundError(f'M53 evidence missing; finish M53 first: {required}')


In [ ]:
# Freeze the M54 config and manifest from the completed M53 evidence.
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run([
    sys.executable,'scripts/prepare_monodgp_m54_adaptation.py',
    '--monodgp-repo',MONODGP_REPO,
    '--dataset-root',DATASET_ROOT,
    '--m53-gate',M53_GATE,
    '--m53-manifest',M53_MANIFEST,
    '--output-root',OUTPUT_ROOT,
],cwd=MOBILE_REPO)
MANIFEST=OUTPUT_ROOT/'m54_adaptation_manifest.json'
manifest=json.loads(MANIFEST.read_text())
CONFIG=Path(manifest['runtime_config'])
assert manifest['training_authorized'] is True
assert manifest['distillation_enabled'] is False
assert manifest['architecture_changed'] is False
assert manifest['training_schedule']['max_epochs']==MAX_EPOCHS
print(json.dumps(manifest,indent=2))


In [ ]:
# Mandatory real CUDA forward/loss/backward/optimizer-step smoke.
SMOKE=OUTPUT_ROOT/'m54_training_smoke.json'
SMOKE_LOG=OUTPUT_ROOT/'colab_logs/m54_training_smoke.log'
SMOKE_LOG.parent.mkdir(parents=True,exist_ok=True)
command=[
    sys.executable,'-u','scripts/smoke_test_monodgp_m54_training.py',
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--output',SMOKE,
]
print('+',shlex.join(map(str,command)),'\nDurable log:',SMOKE_LOG,flush=True)
with SMOKE_LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen([str(x) for x in command],cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line)
    return_code=process.wait()
if return_code: raise RuntimeError(f'M54 smoke exited {return_code}; full log: {SMOKE_LOG}')
smoke=json.loads(SMOKE.read_text())
assert smoke['complete'] and smoke['finite_outputs'] and smoke['finite_gradients']
assert smoke['optimizer_steps']==1 and smoke['vehicle_targets']>0 and smoke['pedestrian_targets']>0
print(json.dumps(smoke,indent=2))
print('STOP HERE and return:',SMOKE)


## Stop point

Return m54_training_smoke.json for review. Do not run the remaining cells until the smoke has passed. The cells below perform the long, restartable 100-epoch training and complete checkpoint qualification.


In [ ]:
# Detect the newest complete Drive checkpoint and build an exact resume config.
import yaml
sys.path.insert(0,str(MONODGP_REPO))
from lib.helpers.save_helper import load_checkpoint_safely
checkpoint_pattern=re.compile(r'^checkpoint_epoch_(\d+)\.pth$')
valid_checkpoints=[]
for path in RUN_DIR.glob('checkpoint_epoch_*.pth'):
    match=checkpoint_pattern.fullmatch(path.name)
    if not match: continue
    try:
        payload=load_checkpoint_safely(path,'cpu')
        epoch=int(payload.get('epoch',-1))
        if epoch!=int(match.group(1)): raise ValueError('filename/payload epoch mismatch')
        if payload.get('model_state') is None: raise ValueError('model_state missing')
        if payload.get('optimizer_state') is None: raise ValueError('optimizer_state missing')
        valid_checkpoints.append((epoch,path))
        print(f'Valid resume checkpoint: epoch={epoch} size={path.stat().st_size/1e6:.1f}MB {path}')
    except Exception as error:
        print(f'Skipping invalid checkpoint {path}: {type(error).__name__}: {error}')
latest=max(valid_checkpoints,default=None,key=lambda item:item[0])
run_cfg=yaml.safe_load(CONFIG.read_text())
if latest is None:
    START_EPOCH=0
    CONFIG_TO_RUN=CONFIG
    print('Starting from the exact verified M53 checkpoint.')
else:
    START_EPOCH,RESUME_CHECKPOINT=latest
    run_cfg['trainer'].pop('pretrain_model',None)
    run_cfg['trainer']['resume_model']=str(RESUME_CHECKPOINT)
    run_cfg['trainer']['max_epoch']=MAX_EPOCHS
    CONFIG_TO_RUN=MONODGP_REPO/'configs/monodgp_m54_vehicle_pedestrian_resume.yaml'
    CONFIG_TO_RUN.write_text(yaml.safe_dump(run_cfg,sort_keys=False))
    print(f'Resuming M54 after completed epoch {START_EPOCH}: {RESUME_CHECKPOINT}')
print('Training config:',CONFIG_TO_RUN)
print('Remaining epochs:',max(0,MAX_EPOCHS-START_EPOCH))


In [ ]:
# Long training with live output and a durable combined Drive log.
from collections import deque
from datetime import datetime, timezone
LOG_DIR=OUTPUT_ROOT/'colab_logs'
LOG_DIR.mkdir(parents=True,exist_ok=True)
def run_training_logged(command,cwd):
    command=[str(x) for x in command]
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    log_path=LOG_DIR/f'train_{RUN_NAME}_{stamp}.log'
    print('+',shlex.join(command),'\nDurable combined log:',log_path,flush=True)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    tail=deque(maxlen=160)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code:
        print('Last captured lines:\n'+('\n'.join(tail) if tail else '<no output>'))
        subprocess.run(['nvidia-smi']); subprocess.run(['df','-h','/content','/content/drive'])
        raise RuntimeError(f'M54 training exited {code}; durable log: {log_path}')
    return log_path
if START_EPOCH>=MAX_EPOCHS:
    print(f'M54 already reached epoch {START_EPOCH}; no training required.')
else:
    TRAIN_LOG=run_training_logged([sys.executable,'-u','tools/train_val.py','--config',CONFIG_TO_RUN],MONODGP_REPO)


In [ ]:
# Restartable complete checkpoint sweep and selected-checkpoint qualification.
SWEEP_DIR=OUTPUT_ROOT/'product_checkpoint_sweep'
run([
    sys.executable,'-u','scripts/sweep_monodgp_m54_product_checkpoints.py',
    '--monodgp-repo',MONODGP_REPO,
    '--mobile-repo',MOBILE_REPO,
    '--manifest',MANIFEST,
    '--dataset-root',DATASET_ROOT,
    '--split-dir',SPLIT_DIR,
    '--output-dir',SWEEP_DIR,
    '--product-config','configs/kitti_mobileadas3d_s1.yaml',
    '--profile','colab_drive',
    '--score-threshold','0.001',
    '--topk','50',
    '--epochs',*map(str,SWEEP_EPOCHS),
],cwd=MOBILE_REPO)
REPORT=SWEEP_DIR/'m54_product_selection.json'
report=json.loads(REPORT.read_text())
print(json.dumps(report,indent=2))
import pandas as pd
display(pd.read_csv(SWEEP_DIR/'m54_product_checkpoint_sweep.csv').head(12))
print('Return:',REPORT)
